In [ ]:
import pandas as pd

df = pd.read_parquet("../Data_Processed/nat_nn_tabular_dataset.parquet")
print(df.shape)
df.head()


(278558, 212)


,player_id,valuation_date,y_raw,y_log,height_in_cm,age_years,is_big5_league,foot_B,foot_L,foot_R,...,cumulative_yellow_cards,cumulative_red_cards,cumulative_sub_in,cumulative_sub_out,lag_10_goals,lag_10_assists,lag_10_yellow_cards,lag_10_red_cards,lag_10_sub_in,lag_10_sub_out
0,10,2013-01-14,4000000.0,15.201805,184.0,34.600958,1.0,0,0,1,...,6.0,0.0,3.0,8.0,4.0,1.0,1.0,0.0,2.0,5.0
1,10,2013-06-19,2000000.0,14.508658,184.0,35.028063,1.0,0,0,1,...,8.0,0.0,6.0,13.0,5.0,1.0,2.0,0.0,3.0,5.0
2,10,2014-01-07,1000000.0,13.815512,184.0,35.581109,1.0,0,0,1,...,9.0,0.0,9.0,16.0,5.0,3.0,2.0,0.0,3.0,3.0
3,10,2014-07-07,1000000.0,13.815512,184.0,36.076660,1.0,0,0,1,...,10.0,0.0,11.0,21.0,3.0,2.0,0.0,0.0,2.0,5.0
4,10,2015-01-07,1000000.0,13.815512,184.0,36.580424,1.0,0,0,1,...,12.0,0.0,21.0,25.0,3.0,1.0,2.0,0.0,7.0,3.0


In [7]:
import sys
from pathlib import Path
import importlib

sys.path.append(str(Path("..").resolve()))

from Implementations import prepare_data
importlib.reload(prepare_data)
from Implementations.prepare_data import prepare_data

data = prepare_data(df, test_size=0.2, seed=42)
print(data["summary"])


{'n_rows_total': 278558, 'n_rows_used': 275230, 'n_features': 208, 'test_size': 0.2, 'seed': 42, 'numeric_only': True, 'drop_na': True, 'standardize': True, 'player_overlap_train_test': 0}


In [8]:
import numpy as np

X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["ylog_train"].ravel()
y_test  = data["ylog_test"].ravel()

print("Shapes:", X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print("Dtypes:", X_train.dtype, X_test.dtype, y_train.dtype, y_test.dtype)

def check_array(name, A):
    A = np.asarray(A)
    print(f"{name}: finite={np.isfinite(A).all()}, min={np.nanmin(A):.3g}, max={np.nanmax(A):.3g}")
    if not np.isfinite(A).all():
        bad = np.where(~np.isfinite(A))
        print(f"  -> {name} has non-finite at first idx:", tuple(b[0] for b in bad))
check_array("X_train", X_train)
check_array("X_test", X_test)
check_array("y_train", y_train)
check_array("y_test", y_test)


Shapes: (219940, 208) (55290, 208) (219940,) (55290,)
Dtypes: float32 float32 float32 float32
X_train: finite=True, min=-25.2, max=469
X_test: finite=True, min=-24.9, max=332
y_train: finite=True, min=9.21, max=19.1
y_test: finite=True, min=9.21, max=19


In [9]:
print("n_features:", X_train.shape[1])
print("Approx RAM for X_train (GB):", X_train.nbytes / 1e9)


n_features: 208
Approx RAM for X_train (GB): 0.18299008


In [10]:
import os
import torch

# Begrens tråder (viktig på Mac)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

# Slå av mkldnn (kan gi segfault i noen builds)
torch.backends.mkldnn.enabled = False


In [11]:
import numpy as np
import torch
from torch import nn
from sklearn.metrics import mean_squared_error, r2_score

# --- data (dere har allerede verifisert at disse er fine)
X_train = np.asarray(data["X_train"], dtype=np.float32)
X_test  = np.asarray(data["X_test"],  dtype=np.float32)
y_train = np.asarray(data["ylog_train"].ravel(), dtype=np.float32)
y_test  = np.asarray(data["ylog_test"].ravel(),  dtype=np.float32)

# (ekstra sikkerhet)
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

device = "cpu"
torch.manual_seed(42)

Xtr = torch.from_numpy(X_train).to(device)
ytr = torch.from_numpy(y_train).unsqueeze(1).to(device)
Xte = torch.from_numpy(X_test).to(device)
yte = torch.from_numpy(y_test).unsqueeze(1).to(device)

class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )
    def forward(self, x): return self.net(x)

model = MLP(d_in=X_train.shape[1]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
loss_fn = nn.MSELoss()

@torch.no_grad()
def eval_model():
    model.eval()
    # eval i chunks (ingen DataLoader)
    bs = 2048
    preds = []
    for i in range(0, Xte.shape[0], bs):
        preds.append(model(Xte[i:i+bs]).cpu().numpy())
    yhat = np.vstack(preds).ravel()
    rmse = np.sqrt(mean_squared_error(y_test, yhat))
    r2 = r2_score(y_test, yhat)
    return rmse, r2

batch_size = 256
n = Xtr.shape[0]

for epoch in range(1, 21):
    model.train()

    # shuffle indices (på CPU)
    perm = torch.randperm(n)
    for i in range(0, n, batch_size):
        idx = perm[i:i+batch_size]
        xb = Xtr[idx]
        yb = ytr[idx]

        pred = model(xb)
        loss = loss_fn(pred, yb)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()

    rmse, r2 = eval_model()
    print(f"Epoch {epoch:02d} | RMSE(log)={rmse:.4f} | R2(log)={r2:.4f}")


Epoch 01 | RMSE(log)=1.0676 | R2(log)=0.4975
Epoch 02 | RMSE(log)=1.0374 | R2(log)=0.5255
Epoch 03 | RMSE(log)=1.0325 | R2(log)=0.5300
Epoch 04 | RMSE(log)=1.0279 | R2(log)=0.5341
Epoch 05 | RMSE(log)=1.0201 | R2(log)=0.5412
Epoch 06 | RMSE(log)=1.0272 | R2(log)=0.5348
Epoch 07 | RMSE(log)=1.0201 | R2(log)=0.5412
Epoch 08 | RMSE(log)=1.0187 | R2(log)=0.5425
Epoch 09 | RMSE(log)=1.0137 | R2(log)=0.5469
Epoch 10 | RMSE(log)=1.0080 | R2(log)=0.5521
Epoch 11 | RMSE(log)=1.0106 | R2(log)=0.5498
Epoch 12 | RMSE(log)=1.0021 | R2(log)=0.5573
Epoch 13 | RMSE(log)=1.0212 | R2(log)=0.5402
Epoch 14 | RMSE(log)=1.0020 | R2(log)=0.5573
Epoch 15 | RMSE(log)=0.9616 | R2(log)=0.5923
Epoch 16 | RMSE(log)=0.9728 | R2(log)=0.5828
Epoch 17 | RMSE(log)=0.9332 | R2(log)=0.6161
Epoch 18 | RMSE(log)=0.9216 | R2(log)=0.6255
Epoch 19 | RMSE(log)=0.9208 | R2(log)=0.6262
Epoch 20 | RMSE(log)=0.9161 | R2(log)=0.6300
